# Urban Delivery : le trio parcimonieux

Ce notebook construit la soumission **`submission_trio_parcimonieux.csv`** : la moyenne pondérée de trois modèles.
1. **S\*\* parcimonieux** : régression structurée T = A(x) + distance × exp(M(x)), delta 8, pénalités de trajet 2,4 / 6 000, courbure en distance ; l'allure M(x) ne garde que véhicule, trafic, heure de pointe, pression et leurs interactions.
2. **S\* delta 8 parcimonieux** : même allure réduite, sans courbure, pénalités par défaut.
3. **Réseau de neurones structuré** (numpy, 10 graines).

Les poids sont choisis par validation imbriquée, puis les trois modèles sont réappris sur janvier-octobre. Il reprend le nettoyage, les variables, les plis et le modèle structuré de `Urban_Delivery_Pistes.ipynb` (perte censurée [8, 210], optimiseur convergé, indicateurs de valeurs manquantes).

## Résultat en bref
| Modèle | MAE (3 plis) | Gain contre S\* [IC 95 %] |
|---|---|---|
| S\* (référence) | 6,4741 | |
| Trio : S\*\* + S\* delta 8 + réseau | 6,4717 | +0,0024 [+0,0009 ; +0,0038] |
| **Trio parcimonieux** | **6,4714** | **+0,0028 [+0,0010 ; +0,0044]** |

Poids finaux : S\*\* parcimonieux 0,598, S\* delta 8 parcimonieux 0,313, réseau 0,089. Le trio parcimonieux fait 0,0004 de mieux que le trio, sans que ce petit écart soit significatif [−0,0004 ; +0,0012].

## 1. Préparation
Placez ce notebook dans le dossier des deux CSV. Bibliothèques : numpy, pandas, scipy, scikit-learn, joblib, matplotlib. Durée : **4 min** environ (2 cœurs). Les prédictions par pli sont gardées dans `cache_trio/` : supprimez ce dossier après toute modification du code.

In [3]:
import os, time, pickle, itertools, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40, 'display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'axes.axisbelow': True, 'font.size': 10})
BLEU, GRIS = '#2a6fbb', '#9a9a9a'

SEED = 42
FICHIER_TRAIN, FICHIER_TEST = 'urban_delivery_train.csv', 'urban_delivery_test.csv'
Y_MIN, Y_MAX = 8, 210        # plafonds observés sur la cible
SEUIL_CIBLE = 215            # au-delà : anomalies x5
B_BOOT = 2000                # tirages du bootstrap apparié
N_JOBS = min(4, os.cpu_count() or 1)
CACHE = os.path.abspath('cache_trio')
os.makedirs(CACHE, exist_ok=True)
T_DEBUT = time.time()
print('pandas', pd.__version__, '| numpy', np.__version__, '| calculs parallèles :', N_JOBS)

pandas 2.2.3 | numpy 1.26.4 | calculs parallèles : 4


## 2. Nettoyage
Identique à la solution.

In [5]:
FORMATS_DATE = [
    (r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', '%Y-%m-%d %H:%M:%S'),
    (r'^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$',       '%d/%m/%Y %H:%M'),
    (r'^\d{4}/\d{2}/\d{2} \d{2}:\d{2}$',       '%Y/%m/%d %H:%M'),
]

def lire_dates(s):
    s = s.astype('string').str.strip()
    sortie = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for motif, fmt in FORMATS_DATE:
        masque = s.str.match(motif).fillna(False).astype(bool)
        sortie[masque] = pd.to_datetime(s[masque], format=fmt, errors='coerce')
    return sortie

def nettoyer(df):
    d = df.copy()
    d['date'] = lire_dates(d['order_datetime'])
    for c in ['pickup_zone', 'delivery_zone']:
        d[c] = d[c].astype('string').str.strip().str.upper().astype(str)
    for c in ['vehicle_type', 'merchant_category', 'traffic_level', 'weather_condition']:
        d[c] = d[c].astype('string').str.strip().str.lower().astype(str)
    v = pd.to_numeric(d['order_value'].astype(str).str.replace(r'[^\d\-]', '', regex=True), errors='coerce')
    d['order_value'] = v.where(v > 0)
    d['package_weight_kg'] = d['package_weight_kg'].where(d['package_weight_kg'] > 0)
    t = d['temperature_c']
    d['temperature_c'] = t.where(t < 50, (t - 32) / 1.8)
    d['courier_rating'] = d['courier_rating'].where(d['courier_rating'].between(1, 5))
    d['available_couriers_in_zone'] = d['available_couriers_in_zone'].where(d['available_couriers_in_zone'] >= 0)
    for c in ['pickup_latitude', 'pickup_longitude', 'delivery_latitude', 'delivery_longitude']:
        d[c] = d[c].where(d[c] != 0)
    return d

train_brut = pd.read_csv(FICHIER_TRAIN)
test_brut = pd.read_csv(FICHIER_TEST)
train, test = nettoyer(train_brut), nettoyer(test_brut)
n0 = len(train)
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'id'])
y = train['target']
base = train[(y > 0) & (y <= SEUIL_CIBLE) & train['date'].notna()].copy()
print(f'Doublons supprimés : {n0 - len(train)} | lignes pour apprendre : {len(base)} | test : {len(test)}')

Doublons supprimés : 150 | lignes pour apprendre : 99841 | test : 19892


## 3. Variables
Identiques, avec les indicateurs de valeurs manquantes.

In [7]:
ORDRE_TRAFIC = {'low': 0, 'medium': 1, 'high': 2, 'severe': 3}
ORDRE_METEO = {'clear': 0, 'cloudy': 1, 'light rain': 2, 'heavy rain': 3}
HEURES_POINTE = [7, 8, 9, 17, 18, 19, 20]

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def centres_zones(d):
    p = d.groupby('pickup_zone')[['pickup_latitude', 'pickup_longitude']].median()
    q = d.groupby('delivery_zone')[['delivery_latitude', 'delivery_longitude']].median()
    p.columns = q.columns = ['lat', 'lon']
    return (p + q) / 2

CENTRES = centres_zones(train)

def ajouter_variables(d):
    d = d.copy()
    for cote, zone in [('pickup', 'pickup_zone'), ('delivery', 'delivery_zone')]:
        d[f'{cote}_latitude'] = d[f'{cote}_latitude'].fillna(d[zone].map(CENTRES['lat']))
        d[f'{cote}_longitude'] = d[f'{cote}_longitude'].fillna(d[zone].map(CENTRES['lon']))
    d['dist_km'] = haversine(d.pickup_latitude, d.pickup_longitude, d.delivery_latitude, d.delivery_longitude)
    d['zone_dist_km'] = haversine(d.pickup_zone.map(CENTRES.lat), d.pickup_zone.map(CENTRES.lon),
                                  d.delivery_zone.map(CENTRES.lat), d.delivery_zone.map(CENTRES.lon))
    d['same_zone'] = (d.pickup_zone == d.delivery_zone).astype(int)
    d['trajet'] = d.pickup_zone + '>' + d.delivery_zone
    d['heure'] = d['date'].dt.hour
    d['peak'] = d['heure'].isin(HEURES_POINTE).astype(float)
    d['dow'] = d['date'].dt.dayofweek
    d['traffic_ord'] = d.traffic_level.map(ORDRE_TRAFIC).astype(float)
    d['weather_ord'] = d.weather_condition.map(ORDRE_METEO).astype(float)
    d['pressure'] = d.active_orders_in_zone / (d.available_couriers_in_zone + 1)
    d['prep_best'] = d.estimated_prep_time.fillna(d.merchant_avg_prep_time)
    d['prep_gap'] = d.estimated_prep_time - d.merchant_avg_prep_time
    return d

COORDS = ['pickup_latitude', 'pickup_longitude', 'delivery_latitude', 'delivery_longitude']

def ajouter_drapeaux(d):
    # Drapeaux calculés avant l'imputation des coordonnées (aucune cible).
    d = d.copy()
    for c in COORDS:
        d[c + '_brut'] = d[c]
    d['coord_imp'] = d[COORDS].isna().any(axis=1).astype(float)
    d['prep_est_nan'] = d['estimated_prep_time'].isna().astype(float)
    d['prep_moy_nan'] = d['merchant_avg_prep_time'].isna().astype(float)
    return d

base = ajouter_variables(ajouter_drapeaux(base))
test_fe = ajouter_variables(ajouter_drapeaux(test))

## 4. Plis, métriques et tests
Plis 1 à 3 pour l'évaluation ; pli 0 pour les réglages. Validation imbriquée : les poids appliqués au pli k sont choisis sur les plis qui le précèdent. Bootstrap apparié stratifié par pli (2 000 tirages).

In [9]:
PLIS = {0: ('2025-03-01', '2025-05-01'), 1: ('2025-05-01', '2025-07-01'),
        2: ('2025-07-01', '2025-09-01'), 3: ('2025-09-01', '2025-11-01')}
EVAL = (1, 2, 3)

def decoupe(d, debut, fin):
    return d[d['date'] < debut], d[(d['date'] >= debut) & (d['date'] < fin)]

def borne(p):
    return np.clip(p, Y_MIN, Y_MAX)

def mae(y, p):
    return float(np.abs(np.asarray(y) - borne(np.asarray(p))).mean())

def rmse(y, p):
    return float(np.sqrt(((np.asarray(y) - borne(np.asarray(p))) ** 2).mean()))

Y = {k: decoupe(base, *PLIS[k])[1]['target'].values for k in PLIS}
JOURS = {k: decoupe(base, *PLIS[k])[1]['date'].dt.normalize().values for k in PLIS}
for k, (debut, fin) in PLIS.items():
    a, v = decoupe(base, debut, fin)
    print(f'Pli {k} : apprentissage {len(a):6d} lignes | validation {len(v):6d} lignes')

# ---------- prédictions par pli, avec cache disque
def _calcule(nom, fonction, plis, donnees):
    P = {}
    for k in plis:
        a, v = decoupe(donnees, *PLIS[k])
        P[k] = np.asarray(fonction(a, v, k), dtype=float)
    with open(os.path.join(CACHE, nom + '.pkl'), 'wb') as f:
        pickle.dump(P, f)
    return P

def predictions(nom, fonction, plis=(0, 1, 2, 3)):
    """{pli: prédictions sur la validation du pli} ; fonction(apprentissage, validation, pli)."""
    chemin = os.path.join(CACHE, nom + '.pkl')
    if os.path.exists(chemin):
        with open(chemin, 'rb') as f:
            P = pickle.load(f)
        if all(k in P for k in plis):
            return P
    return _calcule(nom, fonction, plis, base)

def predictions_paralleles(taches, plis=(0, 1, 2, 3)):
    """taches = {nom: fonction}. Les combinaisons absentes du cache sont calculées en parallèle."""
    a_faire = [n for n in taches if not os.path.exists(os.path.join(CACHE, n + '.pkl'))]
    if a_faire:
        Parallel(n_jobs=N_JOBS)(delayed(_calcule)(n, taches[n], plis, base) for n in a_faire)
    return {n: predictions(n, taches[n], plis) for n in taches}

# ---------- bootstrap apparié stratifié par pli
def bootstrap_gain(P_ref, P_new, plis=EVAL, B=B_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    tirages, gain = np.zeros(B), 0.
    for k in plis:
        d = np.abs(Y[k] - borne(P_ref[k])) - np.abs(Y[k] - borne(P_new[k]))
        gain += d.mean() / len(plis)
        for i in range(0, B, 250):
            b = min(250, B - i)
            tirages[i:i + b] += d[rng.integers(0, len(d), size=(b, len(d)))].mean(axis=1) / len(plis)
    lo, hi = np.percentile(tirages, [2.5, 97.5])
    return gain, lo, hi

def bootstrap_gain_jours(P_ref, P_new, plis=EVAL, B=B_BOOT, seed=SEED):
    """Variante plus prudente : on tire des journées entières (erreurs corrélées dans une journée)."""
    rng = np.random.default_rng(seed)
    tirages, gain = np.zeros(B), 0.
    for k in plis:
        d = np.abs(Y[k] - borne(P_ref[k])) - np.abs(Y[k] - borne(P_new[k]))
        gain += d.mean() / len(plis)
        _, inv = np.unique(JOURS[k], return_inverse=True)
        s, c = np.bincount(inv, weights=d), np.bincount(inv)
        t = rng.integers(0, len(s), size=(B, len(s)))
        tirages += s[t].sum(axis=1) / c[t].sum(axis=1) / len(plis)
    lo, hi = np.percentile(tirages, [2.5, 97.5])
    return gain, lo, hi

# ---------- validation imbriquée
def internes(k):
    return list(range(k))          # k = 4 : modèle final janvier-octobre

def selection_imbriquee(candidats):
    """Pour chaque pli k, choisit le candidat de plus faible MAE moyenne sur les plis internes de k."""
    score = lambda c, k: np.mean([mae(Y[j], candidats[c][j]) for j in internes(k)])
    choix = {k: min(candidats, key=lambda c: score(c, k)) for k in (1, 2, 3, 4)}
    return {k: candidats[choix[k]][k] for k in EVAL}, choix

# ---------- tableau des résultats
TABLEAU = []
def noter(etape, modele, P, ref=None, lecture=False):
    """Ajoute une ligne au tableau. Avec ref : gain et IC 95 %. Renvoie True si la modification est retenue."""
    m = [mae(Y[k], P[k]) for k in EVAL]; r = [rmse(Y[k], P[k]) for k in EVAL]
    ligne = {'étape': etape, 'modèle': modele, 'MAE': np.mean(m), 'RMSE': np.mean(r),
             'pli 1': m[0], 'pli 2': m[1], 'pli 3': m[2], 'gain': np.nan, 'IC bas': np.nan, 'IC haut': np.nan, 'décision': ''}
    retenu = False
    if ref is not None:
        g, lo, hi = bootstrap_gain(ref, P)
        retenu = (lo > 0) and not lecture
        libelle = lecture if isinstance(lecture, str) else 'lecture'
        ligne.update({'gain': g, 'IC bas': lo, 'IC haut': hi,
                      'décision': libelle if lecture else ('retenu' if retenu else 'non retenu')})
    TABLEAU.append(ligne)
    texte = f"{etape:9s} {modele:58s} MAE {ligne['MAE']:.4f} | RMSE {ligne['RMSE']:.4f} | plis {np.round(m, 4)}"
    if ref is not None:
        texte += f" | gain {ligne['gain']:+.4f} [{ligne['IC bas']:+.4f} ; {ligne['IC haut']:+.4f}] -> {ligne['décision']}"
    print(texte)
    return retenu

Pli 0 : apprentissage  19445 lignes | validation  19805 lignes
Pli 1 : apprentissage  39250 lignes | validation  20141 lignes
Pli 2 : apprentissage  59391 lignes | validation  20414 lignes
Pli 3 : apprentissage  79805 lignes | validation  20036 lignes


## 5. Modèle structuré (repris de `Urban_Delivery_Pistes.ipynb`)

In [11]:
# ---------- modèle F (identique à la solution)
NUM_EQUIPE = ['distance_km', 'dist_x_velo', 'dist_x_voiture', 'dist_x_trafic', 'dist_x_pointe', 'trafic_num', 'charge_zone',
              'estimated_prep_time', 'items_count', 'package_weight_kg', 'distance_carre', 'trajet_court', 'prep_x_court',
              'dist_x_pluie', 'velo_x_severe', 'pointe_x_trafic', 'prep_x_charge']
CAT_EQUIPE = ['vehicle_type', 'weather_condition', 'merchant_category', 'heure_cat', 'trajet']

def variables_equipe(d):
    e = d.copy()
    e['distance_km'] = e.dist_km; e['heure_pointe'] = e.peak; e['trafic_num'] = e.traffic_ord; e['charge_zone'] = e.pressure
    velo = e.vehicle_type == 'bicycle'
    e['dist_x_velo'] = e.distance_km * velo
    e['dist_x_voiture'] = e.distance_km * (e.vehicle_type == 'car')
    e['dist_x_trafic'] = e.distance_km * e.trafic_num
    e['dist_x_pointe'] = e.distance_km * e.heure_pointe
    e['distance_carre'] = e.distance_km ** 2
    e['trajet_court'] = (e.distance_km < 5).astype(int)
    e['prep_x_court'] = e.estimated_prep_time * e.trajet_court
    e['dist_x_pluie'] = e.distance_km * (e.weather_condition == 'heavy rain')
    e['velo_x_severe'] = e.distance_km * velo * (e.traffic_level == 'severe')
    e['pointe_x_trafic'] = e.heure_pointe * e.trafic_num
    e['prep_x_charge'] = e.estimated_prep_time * e.charge_zone
    e['heure_cat'] = e.heure.fillna(-1).astype(int).astype(str)
    return e

def modele_equipe(a, v, n_iter=15):
    a, v = variables_equipe(a), variables_equipe(v)
    mediane = a[NUM_EQUIPE].median()
    Xa = pd.concat([a[NUM_EQUIPE].fillna(mediane), pd.get_dummies(a[CAT_EQUIPE], drop_first=True, dtype=float)], axis=1)
    Xv = pd.concat([v[NUM_EQUIPE].fillna(mediane), pd.get_dummies(v[CAT_EQUIPE], dtype=float)], axis=1).reindex(columns=Xa.columns, fill_value=0.)
    poly = PolynomialFeatures(2, include_bias=False).fit(Xa[NUM_EQUIPE])
    cats = [c for c in Xa.columns if c not in NUM_EQUIPE]
    A = np.hstack([poly.transform(Xa[NUM_EQUIPE]), Xa[cats].values])
    B = np.hstack([poly.transform(Xv[NUM_EQUIPE]), Xv[cats].values])
    ech = StandardScaler().fit(A); A, B = ech.transform(A), ech.transform(B)
    y, w = a['target'].values, np.ones(len(a))
    for _ in range(n_iter):
        m = LinearRegression().fit(A, y, sample_weight=w)
        w = 1 / np.maximum(np.abs(y - m.predict(A)), .5)
    return m.predict(B)

# ---------- charnières de préparation (statistiques sans cible, partie apprentissage seulement)
CHARNIERES = ['h_q1', 'h_q1_x_pres', 'h_q2', 'h_q2_x_pres', 'h_q3', 'h_q3_x_pres', 'h_c2', 'h_c4', 'h_c8']

def stats_apprentissage(a):
    prep = a['prep_best'].fillna(a['prep_best'].median())
    return {'q': np.percentile(prep, [25, 50, 75]), 'prep_med': float(prep.median()), 'pres_med': float(a['pressure'].median())}

def variables_charnieres(d, st, cs=(2, 4, 8)):
    """max(0, prép - Qi), max(0, prép - Qi) x pression, max(0, prép - c x pression)."""
    prep = d['prep_best'].fillna(st['prep_med']).values
    pres = d['pressure'].fillna(st['pres_med']).values
    cols = {}
    for i, q in enumerate(st['q']):
        h = np.maximum(0, prep - q)
        cols[f'h_q{i + 1}'], cols[f'h_q{i + 1}_x_pres'] = h, h * pres
    for c in cs:
        cols[f'h_c{c}'] = np.maximum(0, prep - c * pres)
    return cols

# ---------- F' = F + charnières (linéaires) + LAD censuré
def modele_F_plus(a, v, n_iter=15, n_iter_cens=15):
    a, v = variables_equipe(a), variables_equipe(v)
    st = stats_apprentissage(a)
    for d in (a, v):
        for nom, val in variables_charnieres(d, st).items():
            d[nom] = val
    num = NUM_EQUIPE + CHARNIERES
    mediane = a[num].median()
    Xa = pd.concat([a[num].fillna(mediane), pd.get_dummies(a[CAT_EQUIPE], drop_first=True, dtype=float)], axis=1)
    Xv = pd.concat([v[num].fillna(mediane), pd.get_dummies(v[CAT_EQUIPE], dtype=float)], axis=1).reindex(columns=Xa.columns, fill_value=0.)
    poly = PolynomialFeatures(2, include_bias=False).fit(Xa[NUM_EQUIPE])          # degré 2 : variables de F seulement
    autres = [c for c in Xa.columns if c not in NUM_EQUIPE]                        # charnières + indicatrices
    A = np.hstack([poly.transform(Xa[NUM_EQUIPE]), Xa[autres].values])
    B = np.hstack([poly.transform(Xv[NUM_EQUIPE]), Xv[autres].values])
    ech = StandardScaler().fit(A); A, B = ech.transform(A), ech.transform(B)
    y, w = a['target'].values, np.ones(len(a))
    for _ in range(n_iter):                      # LAD par IRLS (comme F)
        m = LinearRegression().fit(A, y, sample_weight=w)
        w = 1 / np.maximum(np.abs(y - m.predict(A)), .5)
    for _ in range(n_iter_cens):                 # LAD censuré : poids nul hors de [8, 210]
        p = m.predict(A)
        w = ((p > Y_MIN) & (p < Y_MAX)) / np.maximum(np.abs(y - borne(p)), .5)
        m = LinearRegression().fit(A, y, sample_weight=w)
    return m.predict(B)

In [12]:
def indicatrices(valeurs, niveaux):
    position = {v: i for i, v in enumerate(niveaux)}
    colonnes = np.array([position.get(v, -1) for v in valeurs])
    lignes = np.nonzero(colonnes >= 0)[0]
    return sp.csr_matrix((np.ones(len(lignes)), (lignes, colonnes[lignes])), shape=(len(valeurs), len(niveaux)))

def colonne(a):
    return sp.csr_matrix(np.asarray(a, dtype=float).reshape(-1, 1))

VEHICULES = ['bicycle', 'car', 'motorbike']
TRAFICS = ['low', 'medium', 'high', 'severe']
METEOS = ['clear', 'cloudy', 'light rain', 'heavy rain']
COMMERCES = ['bakery', 'grocery', 'pharmacy', 'restaurant', 'retail']
TRAJETS = [f'ZONE_{a:02d}>ZONE_{b:02d}' for a in range(1, 13) for b in range(1, 13)]
ZONES = [f'ZONE_{i:02d}' for i in range(1, 13)]


def distance_mediane_imputee(d, ref, n_tirages=400, seed=0):
    # Ligne dont une coordonnée manque : distance médiane quand la coordonnée manquante prend les valeurs observées
    # dans la même zone (partie apprentissage, coordonnées seulement). Les autres lignes sont inchangées.
    rng = np.random.default_rng(seed)
    dist = d['dist_km'].values.copy()
    brut = d[[c + '_brut' for c in COORDS]].values
    reserves = {}
    for cote, zone in [('pickup', 'pickup_zone'), ('delivery', 'delivery_zone')]:
        cols = [f'{cote}_latitude_brut', f'{cote}_longitude_brut']
        ok = ref[cols].notna().all(axis=1)
        for z, g in ref[ok].groupby(zone):
            v = g[cols].values
            reserves[(cote, z)] = v[rng.choice(len(v), min(n_tirages, len(v)), replace=False)]
    for i in np.nonzero(np.isnan(brut).any(axis=1))[0]:
        lat1, lon1, lat2, lon2 = brut[i]
        R1, R2 = reserves[('pickup', d['pickup_zone'].iat[i])], reserves[('delivery', d['delivery_zone'].iat[i])]
        la1 = R1[:, 0] if np.isnan(lat1) else np.full(len(R1), lat1); lo1 = R1[:, 1] if np.isnan(lon1) else np.full(len(R1), lon1)
        la2 = R2[:, 0] if np.isnan(lat2) else np.full(len(R2), lat2); lo2 = R2[:, 1] if np.isnan(lon2) else np.full(len(R2), lon2)
        dist[i] = np.median(haversine(la1, lo1, la2, lo2))
    return dist


class Structure:
    """T = A(x) + distance x exp(M(x)), perte pseudo-Huber, effets de trajet pénalisés.

    Sans option : modèle de la solution (delta=4, l2=2, l2_trajet=60, L-BFGS limité à 600 itérations).
    normalise    : L-BFGS sur colonnes mises à l'échelle (même minimum, meilleur conditionnement)
    charnieres   : charnières de préparation dans A(x)                                   (piste 1a)
    censure      : perte calculée sur la prédiction bornée à [8, 210], bornes lissées    (piste 1b)
    pression_A   : interactions du temps fixe avec la pression                           (piste 4a)
    l2_trajet_A, l2_trajet_M : pénalités séparées des effets de trajet sur A et sur M    (piste 4b)
    manquants    : indicateurs de valeurs manquantes dans A(x)                           (piste 6a)
    dist_mediane : distance médiane imputée quand une coordonnée manque                  (piste 6b)
    zones        : effets de zone de départ et d'arrivée dans A et M                     (piste 6c)
    """

    def __init__(self, delta=4.0, l2=2.0, l2_trajet=60.0, l2_trajet_A=None, l2_trajet_M=None,
                 trajets_A=True, trajets_M=True, charnieres=False, censure=False, lissage=0.5,
                 pression_A=False, normalise=False, maxiter=600, ftol=2.2e-9, gtol=1e-5,
                 manquants=False, dist_mediane=False, zones=False):
        self.delta, self.l2 = delta, l2
        self.l2_trajet_A = l2_trajet / 100 if l2_trajet_A is None else l2_trajet_A    # règle de la solution
        self.l2_trajet_M = l2_trajet * 100 if l2_trajet_M is None else l2_trajet_M
        self.trajets_A, self.trajets_M = trajets_A, trajets_M
        self.charnieres, self.censure, self.lissage, self.pression_A = charnieres, censure, lissage, pression_A
        self.normalise, self.maxiter, self.ftol, self.gtol = normalise, maxiter, ftol, gtol
        self.manquants, self.dist_mediane, self.zones = manquants, dist_mediane, zones

    def _blocs(self, d):
        n = len(d)
        v, t, w, c = [d[k].tolist() for k in ['vehicle_type', 'traffic_level', 'weather_condition', 'merchant_category']]
        pointe = d['peak'].values
        prep = d['prep_best'].fillna(16.6).values - 16.6
        moy = d['merchant_avg_prep_time'].fillna(d['prep_best']).fillna(14.9).values - 14.9
        commun = [
            indicatrices(v, VEHICULES[1:]), indicatrices(t, TRAFICS[1:]), indicatrices(w, METEOS[1:]), colonne(pointe),
            colonne(np.log1p(d['pressure'].fillna(1.9).values) - np.log(2.9)),
            colonne(np.log1p(d['courier_experience_months'].values) - 2.8),
            colonne((d['active_orders_in_zone'].values - 22) / 7),
            colonne((d['available_couriers_in_zone'].fillna(12).values - 12) / 4),
            indicatrices([f'{a}_{b}' for a, b in zip(v, t)], [f'{a}_{b}' for a in VEHICULES[:2] for b in TRAFICS[1:]]),
            indicatrices([f'{b}{int(p)}' for b, p in zip(t, pointe)], [f'{b}1' for b in TRAFICS[1:]]),
            indicatrices([f'{a}{b}' for a, b in zip(w, v)], [f'{a}{b}' for a in METEOS[1:] for b in VEHICULES[:2]]),
            indicatrices([f'{a}{int(p)}' for a, p in zip(c, pointe)], [f'{a}1' for a in COMMERCES]),
        ]
        fixe = [
            colonne(prep), colonne(moy), colonne(d['prep_gap'].fillna(1.7).values - 1.7),
            colonne(prep * (d['merchant_category'].values == 'restaurant')),
            colonne(d['items_count'].values - 3.6), indicatrices(c, COMMERCES[1:]),
            indicatrices(d['heure'].fillna(-1).astype(int).astype(str).tolist(), [str(i) for i in range(1, 24)]),
            colonne(d['same_zone'].values),
        ]
        un = sp.csr_matrix(np.ones((n, 1)))
        blocsA = [('un', un), ('commun', sp.hstack(commun, format='csr')), ('fixe', sp.hstack(fixe, format='csr'))]
        if self.manquants:
            blocsA.append(('manquants', sp.csr_matrix(d[['prep_est_nan', 'prep_moy_nan', 'coord_imp']].values.astype(float))))
        if self.zones:
            zp, zd = indicatrices(d['pickup_zone'].tolist(), ZONES[1:]), indicatrices(d['delivery_zone'].tolist(), ZONES[1:])
            blocsA.append(('zones_A', sp.hstack([zp, zd], format='csr')))
        if self.charnieres:
            blocsA.append(('charnieres', sp.csr_matrix(np.column_stack(list(variables_charnieres(d, self.st_).values())))))
        if self.pression_A:
            lp = np.log1p(d['pressure'].fillna(self.st_['pres_med']).values) - np.log(2.9)
            inter = [prep * lp, pointe * lp, lp ** 2] + [(d['merchant_category'].values == m) * lp for m in COMMERCES[1:]]
            blocsA.append(('pression_A', sp.csr_matrix(np.column_stack(inter))))
        trajet = indicatrices(d['trajet'].tolist(), TRAJETS)
        if self.trajets_A:
            blocsA.append(('trajet_A', trajet))
        blocsM = [('un', un), ('commun', sp.hstack(commun, format='csr'))]
        if self.zones:
            blocsM.append(('zones_M', sp.hstack([zp, zd], format='csr')))
        if self.trajets_M:
            blocsM.append(('trajet_M', trajet))
        dist = distance_mediane_imputee(d, self.ref_) if self.dist_mediane else d['dist_km'].values
        return blocsA, blocsM, dist

    def _matrices(self, d):
        blocsA, blocsM, dist = self._blocs(d)
        reg = np.concatenate([np.full(X.shape[1], {'un': 0., 'trajet_A': self.l2_trajet_A}.get(nom, self.l2)) for nom, X in blocsA]
                             + [np.full(X.shape[1], {'un': 0., 'trajet_M': self.l2_trajet_M}.get(nom, self.l2 * 300)) for nom, X in blocsM])
        return sp.hstack([X for _, X in blocsA], format='csr'), sp.hstack([X for _, X in blocsM], format='csr'), dist, reg

    @staticmethod
    def _prevision(th, XA, XM, dist):
        a = XA @ th[:XA.shape[1]]
        e = np.exp(np.clip(XM @ th[XA.shape[1]:], -3, 3))
        return a + dist * e, e

    def depart_defaut(self, na, nm):
        th = np.zeros(na + nm); th[0] = 24; th[na] = np.log(4)
        return th

    def fit(self, d, y, depart=None):
        y = np.asarray(y, dtype=float)
        self.st_ = stats_apprentissage(d)
        if self.dist_mediane:      # coordonnées observées de la partie apprentissage (sans cible)
            self.ref_ = d[['pickup_zone', 'delivery_zone'] + [c + '_brut' for c in COORDS]]
        XA, XM, dist, reg = self._matrices(d)
        na, nm = XA.shape[1], XM.shape[1]
        n, dl, s = len(y), self.delta, self.lissage
        if self.normalise:     # theta = phi / échelle : même problème, Hessien mieux conditionné
            ea = np.sqrt(np.asarray(XA.multiply(XA).mean(axis=0)).ravel())
            dm = sp.diags(dist) @ XM
            em = 4 * np.sqrt(np.asarray(dm.multiply(dm).mean(axis=0)).ravel())
            ech = np.r_[ea, em]; ech[ech == 0] = 1.
        else:
            ech = np.ones(na + nm)

        def perte(phi):
            th = phi / ech
            p, e = self._prevision(th, XA, XM, dist)
            if self.censure:   # borne lissée : 8 + s.softplus((p - 8)/s) - s.softplus((p - 210)/s)
                u, w = (p - Y_MIN) / s, (p - Y_MAX) / s
                pc = Y_MIN + s * (np.logaddexp(0, u) - np.logaddexp(0, w))
                dpc = 1 / (1 + np.exp(-u)) - 1 / (1 + np.exp(-w))
            else:
                pc, dpc = p, 1.
            r = pc - y
            q = np.sqrt(r * r + dl * dl)
            valeur = (q - dl).sum() / n + .5 * (reg * th * th).sum() / n
            g = r / q / n * dpc
            return valeur, (np.r_[XA.T @ g, XM.T @ (g * dist * e)] + reg * th / n) / ech

        th0 = self.depart_defaut(na, nm) if depart is None else depart
        self.res = minimize(perte, th0 * ech, jac=True, method='L-BFGS-B',
                            options=dict(maxiter=self.maxiter, maxcor=30, ftol=self.ftol, gtol=self.gtol, maxfun=max(15000, 2 * self.maxiter)))
        self.th, self.na_ = self.res.x / ech, na
        return self

    def predict(self, d):
        XA, XM, dist, _ = self._matrices(d)
        return self._prevision(self.th, XA, XM, dist)[0]


def ajuste_structure(a, hp):
    """Avec la perte censurée (non convexe), départ = solution non censurée, comme pour le LAD censuré."""
    if hp.get('censure'):
        m0 = Structure(**{**hp, 'censure': False}).fit(a, a['target'])
        return Structure(**hp).fit(a, a['target'], depart=m0.th.copy())
    return Structure(**hp).fit(a, a['target'])

def f_structure(hp_par_pli, **modif):
    """Fonction de pli : réglages du pli k (hp_par_pli[k]) + modification testée."""
    hp = {k: {**v, **modif} for k, v in hp_par_pli.items()}
    return lambda a, v, k: ajuste_structure(a, hp[k]).predict(v)

## 6. Les composantes du trio
- **Courbure en distance** : log(1 + d) et son carré, dans A et dans M.
- **Allure parcimonieuse** : on retire de M les 17 effets quasi nuls (météo, interactions météo × véhicule, commerce × heure de pointe, expérience du livreur, commandes actives, livreurs disponibles). La météo reste dans le temps fixe A, où elle agit vraiment.
- **Réseau de neurones** : même forme A + d × exp(M), deux couches cachées (32 et 16 neurones), plongement des 144 trajets, perte pseudo-Huber censurée, Adam, arrêt précoce ; moyenne de 10 graines.

S\* (modèle retenu du notebook des pistes) et les versions non parcimonieuses servent de références.

In [14]:
OPT = dict(normalise=True, maxiter=4000, ftol=1e-13, gtol=1e-8)
HP_S = dict(OPT, censure=True, manquants=True)
GARDE_M = [0, 1, 2, 3, 4, 8, 9] + list(range(13, 22))   # colonnes « commun » gardées dans M : véhicule, trafic, pointe, pression, interactions

class StructureCourbe(Structure):
    def _blocs(self, d):
        blocsA, blocsM, dist = super()._blocs(d)
        ld = np.log1p(d['dist_km'].values) - 2.0
        cols = sp.csr_matrix(np.column_stack([ld, ld ** 2]))
        blocsA.insert(3, ('courbe_A', cols)); blocsM.insert(2, ('courbe_M', cols))
        return blocsA, blocsM, dist

class Parci(Structure):
    def _blocs(self, d):
        bA, bM, dist = super()._blocs(d)
        return bA, [(n, X[:, GARDE_M] if n == 'commun' else X) for n, X in bM], dist

class ParciCourbe(StructureCourbe):
    def _blocs(self, d):
        bA, bM, dist = super()._blocs(d)
        return bA, [(n, X[:, GARDE_M] if n == 'commun' else X) for n, X in bM], dist

def ajuste_classe(classe, a, hp):
    m0 = classe(**{**hp, 'censure': False}).fit(a, a['target'])
    return classe(**hp).fit(a, a['target'], depart=m0.th.copy())

def f_classe(classe, **modif):
    hp = {**HP_S, **modif}
    return lambda a, v, k: ajuste_classe(classe, a, hp).predict(v)

def _relu(x):
    return np.maximum(x, 0.)

class ReseauStructure:
    def __init__(self, h1=64, h2=32, emb=6, lr=2e-3, epochs=60, batch=512, delta=4.0, lissage=.5, wd=1e-5,
                 patience=6, seed=0, decroissance=.96):
        self.__dict__.update(dict(h1=h1, h2=h2, emb=emb, lr=lr, epochs=epochs, batch=batch, delta=delta,
                                  lissage=lissage, wd=wd, patience=patience, seed=seed, decroissance=decroissance))

    # ------------------------------------------------------------ variables
    def _brut(self, d):
        h = d['heure'].fillna(12).values; w = d['dow'].fillna(3).values
        cols = {
            'dist': d['dist_km'].values, 'ldist': np.log1p(d['dist_km'].values), 'zdist': d['zone_dist_km'].values,
            'same': d['same_zone'].values, 'hs': np.sin(2 * np.pi * h / 24), 'hc': np.cos(2 * np.pi * h / 24),
            'peak': d['peak'].values, 'ws': np.sin(2 * np.pi * w / 7), 'wc': np.cos(2 * np.pi * w / 7),
            'lpres': np.log1p(d['pressure'].values), 'prep': d['prep_best'].values, 'gap': d['prep_gap'].values,
            'est': d['estimated_prep_time'].values, 'moy': d['merchant_avg_prep_time'].values, 'items': d['items_count'].values,
            'poids': d['package_weight_kg'].values, 'lval': np.log1p(d['order_value'].values), 'temp': d['temperature_c'].values,
            'lexp': np.log1p(d['courier_experience_months'].values), 'note': d['courier_rating'].values,
            'liv30': d['courier_deliveries_30d'].values, 'actifs': d['active_orders_in_zone'].values,
            'dispo': d['available_couriers_in_zone'].values, 'f1': d['prep_est_nan'].values, 'f2': d['prep_moy_nan'].values,
            'f3': d['coord_imp'].values, 'plat': d['pickup_latitude'].values, 'plon': d['pickup_longitude'].values,
            'dlat': d['delivery_latitude'].values, 'dlon': d['delivery_longitude'].values,
        }
        for c, niveaux in [('vehicle_type', VEHICULES), ('traffic_level', TRAFICS), ('weather_condition', METEOS), ('merchant_category', COMMERCES)]:
            for niv in niveaux:
                cols[f'{c}={niv}'] = (d[c].values == niv).astype(float)
        X = np.column_stack([np.asarray(v, dtype=float) for v in cols.values()])
        return X

    def _prep(self, d, fit=False):
        X = self._brut(d)
        if fit:
            self.med_ = np.nanmedian(X, axis=0)
            Xf = np.where(np.isnan(X), self.med_, X)
            self.mu_, self.sd_ = Xf.mean(0), Xf.std(0) + 1e-6
        X = np.where(np.isnan(X), self.med_, X)
        X = (X - self.mu_) / self.sd_
        pos = {t: i for i, t in enumerate(TRAJETS)}
        r = np.array([pos.get(t, 0) for t in d['trajet'].values])
        return X, r, d['dist_km'].values

    # ------------------------------------------------------------ réseau
    def _init(self, nx, rng):
        P = {'E': rng.normal(0, .1, (144, self.emb)),
             'W1': rng.normal(0, np.sqrt(2 / (nx + self.emb)), (nx + self.emb, self.h1)), 'b1': np.zeros(self.h1),
             'W2': rng.normal(0, np.sqrt(2 / self.h1), (self.h1, self.h2)), 'b2': np.zeros(self.h2),
             'wa': rng.normal(0, .01, self.h2), 'ba': np.array(24.), 'wm': rng.normal(0, .001, self.h2), 'bm': np.array(np.log(4.))}
        return P

    @staticmethod
    def _avant(P, X, r, dist):
        z0 = np.hstack([X, P['E'][r]])
        a1 = z0 @ P['W1'] + P['b1']; h1 = _relu(a1)
        a2 = h1 @ P['W2'] + P['b2']; h2 = _relu(a2)
        A = h2 @ P['wa'] + P['ba']; m = h2 @ P['wm'] + P['bm']
        mc = np.clip(m, -3, 3); e = np.exp(mc)
        return A + dist * e, (z0, h1, h2, e, (m > -3) & (m < 3))

    def _perte_grad(self, P, X, r, dist, y):
        p, (z0, h1, h2, e, libre) = self._avant(P, X, r, dist)
        s, dl = self.lissage, self.delta
        u, w = (p - Y_MIN) / s, (p - Y_MAX) / s
        pc = Y_MIN + s * (np.logaddexp(0, u) - np.logaddexp(0, w)); dpc = 1 / (1 + np.exp(-u)) - 1 / (1 + np.exp(-w))
        res = pc - y; q = np.sqrt(res * res + dl * dl)
        n = len(y); L = (q - dl).mean()
        gp = res / q * dpc / n
        gA = gp; gm = gp * dist * e * libre
        G = {'wa': h2.T @ gA, 'ba': gA.sum(), 'wm': h2.T @ gm, 'bm': gm.sum()}
        dh2 = np.outer(gA, P['wa']) + np.outer(gm, P['wm']); dh2[h2 <= 0] = 0
        G['W2'] = h1.T @ dh2; G['b2'] = dh2.sum(0)
        dh1 = dh2 @ P['W2'].T; dh1[h1 <= 0] = 0
        G['W1'] = z0.T @ dh1; G['b1'] = dh1.sum(0)
        dz0 = dh1 @ P['W1'].T
        G['E'] = np.zeros_like(P['E']); np.add.at(G['E'], r, dz0[:, -self.emb:])
        for k in ('W1', 'W2', 'E'):
            L += .5 * self.wd * (P[k] ** 2).sum(); G[k] += self.wd * P[k]
        return L, G

    def _entraine(self, X, r, dist, y, epochs, rng, Xv=None, rv=None, dv=None, yv=None):
        P = self._init(X.shape[1], rng)
        M = {k: np.zeros_like(v) for k, v in P.items()}; V = {k: np.zeros_like(v) for k, v in P.items()}
        t, meilleur, meilleur_ep, attente, histo = 0, np.inf, 0, 0, []
        Pbest = None
        for ep in range(epochs):
            lr = self.lr * self.decroissance ** ep
            idx = rng.permutation(len(y))
            for i in range(0, len(y), self.batch):
                j = idx[i:i + self.batch]
                _, G = self._perte_grad(P, X[j], r[j], dist[j], y[j])
                t += 1
                for k in P:
                    M[k] = .9 * M[k] + .1 * G[k]; V[k] = .999 * V[k] + .001 * G[k] ** 2
                    P[k] = P[k] - lr * (M[k] / (1 - .9 ** t)) / (np.sqrt(V[k] / (1 - .999 ** t)) + 1e-8)
            if Xv is not None:
                pv = self._avant(P, Xv, rv, dv)[0]
                score = np.abs(yv - borne(pv)).mean(); histo.append(score)
                if score < meilleur - 1e-4:
                    meilleur, meilleur_ep, attente = score, ep + 1, 0
                    Pbest = {k: v.copy() for k, v in P.items()}
                else:
                    attente += 1
                    if attente >= self.patience:
                        break
        return (Pbest if Pbest is not None else P), meilleur_ep, histo

    def fit(self, d, y):
        y = np.asarray(y, dtype=float)
        X, r, dist = self._prep(d, fit=True)
        rng = np.random.default_rng(self.seed)
        coupe = d['date'].quantile(.8); ia = (d['date'] < coupe).values; iv = ~ia
        _, self.ep_, self.histo_ = self._entraine(X[ia], r[ia], dist[ia], y[ia], self.epochs, rng, X[iv], r[iv], dist[iv], y[iv])
        self.P_, _, _ = self._entraine(X, r, dist, y, max(self.ep_, 3), np.random.default_rng(self.seed + 1000))
        return self

    def predict(self, d):
        X, r, dist = self._prep(d)
        return self._avant(self.P_, X, r, dist)[0]


CONF_RESEAU = dict(h1=32, h2=16, emb=4, lr=2e-3, wd=1e-4, patience=8)
def f_reseau(a, v, k):
    return np.mean([ReseauStructure(seed=s, **CONF_RESEAU).fit(a, a['target']).predict(v) for s in range(10)], axis=0)

MODELES = {
    'S_etoile': ('S* (référence)', f_classe(Structure)),
    'S2': ('S** = delta 8 + pénalités 4b + courbure', f_classe(StructureCourbe, delta=8., l2_trajet_A=2.4, l2_trajet_M=6000.)),
    'S_d8': ('S* delta 8', f_classe(Structure, delta=8.)),
    'PA_S2': ('S** parcimonieux', f_classe(ParciCourbe, delta=8., l2_trajet_A=2.4, l2_trajet_M=6000.)),
    'PA_d8': ('S* delta 8 parcimonieux', f_classe(Parci, delta=8.)),
    'NN': ('Réseau de neurones structuré (10 graines)', f_reseau),
}
t0 = time.time()
P = predictions_paralleles({n: f for n, (_, f) in MODELES.items()})
print(f'({time.time() - t0:.0f} s)')
REF = P['S_etoile']
_ = noter('Réf.', MODELES['S_etoile'][0], REF)
for n in ['S2', 'S_d8', 'PA_S2', 'PA_d8', 'NN']:
    _ = noter('Modèle', MODELES[n][0], P[n], ref=REF, lecture=True)

(1363 s)
Réf.      S* (référence)                                             MAE 6.4741 | RMSE 9.7094 | plis [6.5491 6.4102 6.4631]
Modèle    S** = delta 8 + pénalités 4b + courbure                    MAE 6.4724 | RMSE 9.7026 | plis [6.5488 6.4081 6.4602] | gain +0.0018 [-0.0003 ; +0.0037] -> lecture
Modèle    S* delta 8                                                 MAE 6.4732 | RMSE 9.7039 | plis [6.5487 6.4094 6.4615] | gain +0.0009 [-0.0000 ; +0.0018] -> lecture
Modèle    S** parcimonieux                                           MAE 6.4720 | RMSE 9.7025 | plis [6.5479 6.4076 6.4606] | gain +0.0021 [-0.0002 ; +0.0043] -> lecture
Modèle    S* delta 8 parcimonieux                                    MAE 6.4728 | RMSE 9.7039 | plis [6.5477 6.409  6.4619] | gain +0.0013 [-0.0001 ; +0.0026] -> lecture
Modèle    Réseau de neurones structuré (10 graines)                  MAE 6.5045 | RMSE 9.7328 | plis [6.5955 6.4413 6.4767] | gain -0.0304 [-0.0374 ; -0.0235] -> lecture


Modèle    S* delta 8                                                 MAE 6.4732 | RMSE 9.7039 | plis [6.5487 6.4094 6.4615] | gain +0.0009 [-0.0000 ; +0.0018] -> lecture


Modèle    S** parcimonieux                                           MAE 6.4720 | RMSE 9.7025 | plis [6.5479 6.4076 6.4606] | gain +0.0021 [-0.0002 ; +0.0043] -> lecture


Modèle    S* delta 8 parcimonieux                                    MAE 6.4728 | RMSE 9.7039 | plis [6.5477 6.409  6.4619] | gain +0.0013 [-0.0001 ; +0.0026] -> lecture


Modèle    Réseau de neurones structuré (10 graines)                  MAE 6.5045 | RMSE 9.7328 | plis [6.5955 6.4413 6.4767] | gain -0.0304 [-0.0374 ; -0.0235] -> lecture


## 7. Les poids du trio
Poids positifs de somme 1, qui minimisent la MAE (lissée sur 0,05 min) des plis internes. Les deux trios sont comparés à S\*, puis entre eux.

In [16]:
def poids_simplexe(noms, plis, dl=.05):
    Pm = np.vstack([np.column_stack([borne(P[n][j]) for n in noms]) for j in plis])
    y = np.concatenate([Y[j] for j in plis])
    def f(z):
        w = np.exp(z - z.max()); w /= w.sum()
        r = Pm @ w - y; q = np.sqrt(r * r + dl * dl)
        g = Pm.T @ (r / q) / len(y)
        return (q - dl).mean(), w * (g - w @ g)
    z = minimize(f, np.zeros(len(noms)), jac=True, method='L-BFGS-B').x
    w = np.exp(z - z.max()); return w / w.sum()

def trio(noms):
    W = {k: poids_simplexe(noms, internes(k)) for k in (1, 2, 3, 4)}
    return {k: np.column_stack([borne(P[n][k]) for n in noms]) @ W[k] for k in EVAL}, W

TRIO, W_TRIO = trio(['S2', 'S_d8', 'NN'])
TRIO_PA, W_PA = trio(['PA_S2', 'PA_d8', 'NN'])
display(pd.DataFrame({('final' if k == 4 else f'pli {k}'): W_PA[k] for k in W_PA},
                     index=[MODELES[n][0] for n in ['PA_S2', 'PA_d8', 'NN']]).round(3))
_ = noter('Trio', 'trio : S** + S* delta 8 + réseau', TRIO, ref=REF, lecture=True)
_ = noter('Trio', 'trio parcimonieux', TRIO_PA, ref=REF, lecture=True)
_ = noter('Écart', 'trio parcimonieux comparé au trio', TRIO_PA, ref=TRIO, lecture=True)
g, lo, hi = bootstrap_gain_jours(REF, TRIO_PA)
print(f'Trio parcimonieux contre S*, bootstrap par journées entières : gain {g:+.4f} [{lo:+.4f} ; {hi:+.4f}]')

,pli 1,pli 2,pli 3,final
S** parcimonieux,0.597,0.544,0.590,0.598
S* delta 8 parcimonieux,0.335,0.395,0.349,0.313
Réseau de neurones structuré (10 graines),0.069,0.061,0.061,0.089


Trio      trio : S** + S* delta 8 + réseau                           MAE 6.4717 | RMSE 9.7020 | plis [6.548  6.4078 6.4594] | gain +0.0024 [+0.0009 ; +0.0038] -> lecture
Trio      trio parcimonieux                                          MAE 6.4714 | RMSE 9.7020 | plis [6.5471 6.4074 6.4597] | gain +0.0028 [+0.0010 ; +0.0044] -> lecture
Écart     trio parcimonieux comparé au trio                          MAE 6.4714 | RMSE 9.7020 | plis [6.5471 6.4074 6.4597] | gain +0.0004 [-0.0004 ; +0.0012] -> lecture
Trio parcimonieux contre S*, bootstrap par journées entières : gain +0.0028 [+0.0008 ; +0.0047]


Écart     trio parcimonieux comparé au trio                          MAE 6.4714 | RMSE 9.7020 | plis [6.5471 6.4074 6.4597] | gain +0.0004 [-0.0004 ; +0.0012] -> lecture
Trio parcimonieux contre S*, bootstrap par journées entières : gain +0.0028 [+0.0008 ; +0.0047]


## 8. Entraînement final (janvier-octobre) et soumission
Les trois composantes sont réapprises sur toutes les lignes propres et combinées avec les poids choisis sur les plis 0 à 3.

In [18]:
t0 = time.time()
w = W_PA[4]
preds = [borne(MODELES[n][1](base, test_fe, 4)) for n in ['PA_S2', 'PA_d8', 'NN']]
p_test = borne(sum(wi * pi for wi, pi in zip(w, preds)))
soumission = pd.DataFrame({'id': test_brut['id'], 'prediction': p_test})
soumission.to_csv('submission_trio_parcimonieux.csv', index=False)
s = pd.read_csv('submission_trio_parcimonieux.csv')
assert len(s) == len(test_brut) and s['id'].is_unique and set(s['id']) == set(test_brut['id'])
assert s['prediction'].notna().all() and s['prediction'].between(Y_MIN, Y_MAX).all()
print('Poids finaux :', dict(zip(['S** parcimonieux', 'S* delta 8 parcimonieux', 'réseau'], np.round(w, 3))))
print('submission_trio_parcimonieux.csv', s.shape, '| moyenne', round(s['prediction'].mean(), 2), '| médiane', round(s['prediction'].median(), 2),
      f'({time.time() - t0:.0f} s)')
print(f'Durée totale du notebook : {(time.time() - T_DEBUT) / 60:.1f} min')

Poids finaux : {'S** parcimonieux': 0.598, 'S* delta 8 parcimonieux': 0.313, 'réseau': 0.089}
submission_trio_parcimonieux.csv (19892, 2) | moyenne 65.82 | médiane 59.1 (701 s)
Durée totale du notebook : 35.1 min


## 9. Conclusion
- **Le trio parcimonieux est le meilleur fichier obtenu** : 6,4714 en validation, soit 0,0028 de mieux que S\*, gain significatif aussi par journées entières.
- **Pourquoi il marche** : les deux régressions structurées captent la loi de simulation ; retirer de l'allure les effets quasi nuls réduit l'erreur d'estimation ; le réseau apporte une petite diversité qui compense une partie des erreurs restantes.
- **Limite** : les effets retirés de l'allure ont été choisis en regardant les coefficients estimés sur les dix mois. Le gain de 0,0004 sur le trio peut donc être légèrement optimiste.
- **Plancher** : environ 6,45 à 6,46 ; il reste au mieux 0,01 à gagner, et l'essentiel de l'erreur est le bruit propre à chaque commande.